# Checkpoint 12 — Compare models on validation data

**What notebook 11 established:** 1,026 training rows (77 positives), 306 validation rows (21 positives), disjoint shipment groups, valid outcome cutoffs, and missing trend values requiring preprocessing. These counts describe this input and are not coded as assumptions.

We now compare five fixed, small candidate configurations. We keep the candidate settings fixed and leave the test set out of model selection. Each model learns preprocessing using the same training rows and predicts the same validation rows.

- Constant: training incident frequency, ignores the features.
- Logistic regression: weighted features mapped to probability; regularized and scaled.
- Shallow tree: at most three levels, with at least 20 training rows per leaf.
- Random forest: averages 200 constrained trees.
- Gradient boosting: builds 100 small trees sequentially to improve its loss; internal random validation/early stopping is disabled to preserve our explicit chronology.

**Selection rule declared before results:** candidates must beat the constant in both average precision and Brier score. Choose the simplest candidate within 0.002 Brier of the best eligible score; otherwise keep the baseline. This tolerance is an explicit engineering preference, not a significance test. Simplicity order: logistic, shallow tree, forest, boosting.

Average precision assesses ranking of incidents; higher is better. Brier and log loss assess probability errors; lower is better. Precision/recall at a fixed 20% threshold illustrate alert tradeoffs. The threshold is neither tuned nor a deployment recommendation.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/dispatch_risk/contracts.py').is_file())
sys.path.insert(0, str(ROOT / 'notebooks' / 'support'))
import workflow as wf
import pandas as pd
import numpy as np
from IPython.display import display
comparison, selection, fitted, train, validation, table, reports = wf.validation_run()
display(comparison)
print(comparison.to_string(index=False))
print("Selected from validation:",selection["selected_model"])
wf.save_json(ROOT / "outputs" / "notebook_results" / "validation_selection.json", selection)
comparison.to_csv(ROOT / "data" / "tables" / "checkpoint12_validation_metrics.csv",index=False)


                 model  rows  ...  true_negative  false_negative
0             constant   306  ...            285              21
1  logistic_regression   306  ...            284               1
2         shallow_tree   306  ...            285               2
3        random_forest   306  ...            284               2
4    gradient_boosting   306  ...            285               2

[5 rows x 12 columns]
              model  rows  positives  average_precision    brier  log_loss  precision_at_0_2  recall_at_0_2  true_positive  false_positive  true_negative  false_negative
           constant   306         21           0.068627 0.063959  0.250379               NaN       0.000000              0               0            285              21
logistic_regression   306         21           0.963370 0.005328  0.030781          0.952381       0.952381             20               1            284               1
       shallow_tree   306         21           0.936989 0.006474  0.036242   

/Users/tirthcshah/Desktop/Tirth Shah/Jobs/FullTime/NextEra Energy/candidate/.venv/lib/python3.14/site-packages/joblib/externals/loky/backend/context.py:134: UserWarning: Could not find the number of physical cores for the following reason:
invalid literal for int() with base 10: ''
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/Users/tirthcshah/Desktop/Tirth Shah/Jobs/FullTime/NextEra Energy/candidate/.venv/lib/python3.14/site-packages/joblib/externals/loky/backend/context.py", line 264, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_darwin()
  File "/Users/tirthcshah/Desktop/Tirth Shah/Jobs/FullTime/NextEra Energy/candidate/.venv/lib/python3.14/site-packages/joblib/externals/loky/backend/context.py", line 420, in _count_physical_cores_darwin
    return int(cpu_info)


## 1. Preserve validation predictions and inspect errors

Save every candidate's probabilities with validation outcomes. This enables later auditing. Rows with the largest probability error illustrate failures, not a representative population sample. Do not change the declared configurations based on final test outcomes.


In [2]:
predictions = validation[["shipment_id","decision_time","label"]].copy()
for name,model in fitted.items():
    predictions[name] = model.predict_proba(validation[wf.FEATURES])[:,1]
predictions.to_csv(ROOT / "data" / "tables" / "checkpoint12_validation_predictions.csv",index=False)
winner = selection["selected_model"]
errors = predictions.assign(absolute_error=(predictions[winner]-predictions.label).abs())
display(errors.sort_values("absolute_error",ascending=False).head(8))
assert np.allclose(predictions["constant"],train.label.mean())
assert set(train.shipment_id).isdisjoint(validation.shipment_id)
assert winner in fitted
print("Baseline frequency and cohort separation verified. Test outcomes not used for selection.")


     shipment_id             decision_time  ...  gradient_boosting  absolute_error
1333     s-00444 2026-02-25 23:00:00+00:00  ...           0.031312        0.997168
1377     s-00458 2026-02-27 20:00:00+00:00  ...           0.108628        0.644960
1240     s-00413 2026-02-22 02:00:00+00:00  ...           0.046698        0.258117
1188     s-00395 2026-02-19 23:00:00+00:00  ...           0.983111        0.203467
1249     s-00416 2026-02-22 11:00:00+00:00  ...           0.046698        0.182889
1331     s-00444 2026-02-25 20:00:00+00:00  ...           0.006480        0.131308
1263     s-00420 2026-02-23 02:00:00+00:00  ...           0.050972        0.094518
1282     s-00427 2026-02-23 20:00:00+00:00  ...           0.056582        0.093540

[8 rows x 9 columns]
Baseline frequency and cohort separation verified. Test outcomes not used for selection.


## 2. Interpret results before notebook 13

A winner on validation is not proven best in general. This validation period has few positive examples; small differences can be unstable. Any apparently excellent performance also reflects the generator's strong synthetic temperature signal. It is not evidence of real-world cold-chain performance.

Notebook 13 will freeze this selection, refit that configuration on eligible development shipments using reports available at test start, and evaluate it once against the constant on the untouched test cohort. This refit is justified because validation choices are complete by test start under the 48-hour maturity rule. It may recover previously immature training rows; test shipments remain excluded.

The final report will include operational slices, reliability bins, and approximate shipment-bootstrap uncertainty. We will not replace the winner because another model might do better on test.

References: [average precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html), [pipeline leakage guidance](https://scikit-learn.org/stable/common_pitfalls.html).

**Try explaining this:** why do we use validation results to choose the model and keep test results for the final check?
